# Exercise 02, Part 3: Flagging bad GNSS signals with a classifier

Run this in **Google Colab**, signed in with your WashU Google account. Nothing
installs on your own computer.

A GNSS receiver assumes by default each satellite signal traveled in a straight line to the
antenna. Near buildings and under tree canopy a signal can bounce off a surface
first, arriving late and making the satellite look farther away than it is (multipath). A
position built from bounced signals is inaccurate, often by meters. This model
reads a RINEX file and, for every satellite at every epoch, predicts whether that
signal arrived cleanly (LOS, line-of-sight) or bounced (NLOS,
non-line-of-sight).

You'll run this model on the same static GNSS collection that you processed with NRCan and Emlid Studio. It will produce a file that you can then rerun, and see if the result changed.



---
### How to run a Colab notebook
Click a cell, then press **Shift+Enter** to run it and move to the next one. Run
the cells in order from the top. A spinning circle at the left means the cell is
still working; wait for it to finish before running the next.

## Step 1: Install pyrtklib

Colab already has `pandas`, `numpy`, `scikit-learn` and `matplotlib`. This takes about 20 seconds.

In [ ]:
!pip install --quiet pyrtklib
import pyrtklib, sklearn, sys
print("python     ", sys.version.split()[0])
print("pyrtklib   ", "ok")
print("scikit-learn", sklearn.__version__)

## Step 2: Fetch the feature extractor and the training data

The notebook automatically downloads two files from GitHub: 
`extract_features.py` turns a RINEX file into the five numbers the model uses.
`nlos_features.csv.gz` is the labeled training table: 26,522 satellite
observations from Hong Kong with a known LOS/NLOS answer for each.

You will train the model here rather than loading a pre-trained one. The training set is small so training is fast.

In [ ]:
import urllib.request, pathlib

BASE = "https://raw.githubusercontent.com/fossettlab/gnss-nlos-notebook/main"
for name, dest in [
    ("extract_features.py", "extract_features.py"),
    ("nlos_features.csv.gz",       "nlos_features.csv.gz"),
]:
    urllib.request.urlretrieve(f"{BASE}/{name}", dest)
    print(f"{dest:26s} {pathlib.Path(dest).stat().st_size/1e6:7.2f} MB")

## Step 3: Train the classifier and check it on data it has not seen

The training data covers four separate collection runs. To get an honest score
we hold out one entire run, train on the other three, and test on the held-out
one, repeating for each run. This is **leave-one-run-out**.

It matters that we hold out a whole run rather than shuffling rows: observations
seconds apart are nearly identical, so a random split would let the model train
on data adjacent to its own test set and report a flattering score.

In [ ]:
import time, pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

FEATURES = ["cn0", "elev_deg", "pr_resid_norm", "pr_rate_consistency", "cn0_std"]
train = pd.read_csv("nlos_features.csv.gz")
print(f"{len(train):,} labeled observations, {train['nlos'].mean():.1%} of them NLOS, "
      f"{train['run'].nunique()} runs")

def make_model():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("clf",    RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                          random_state=42, n_jobs=-1)),
    ])

t0 = time.perf_counter()
scores = []
for held_out in sorted(train["run"].unique()):
    te = train[train["run"] == held_out]
    tr = train[train["run"] != held_out]
    m = make_model().fit(tr[FEATURES], tr["nlos"])
    prob = m.predict_proba(te[FEATURES])[:, 1]
    pred = (prob >= 0.5).astype(int)
    scores.append({"held out": held_out, "n": len(te),
                   "accuracy": accuracy_score(te["nlos"], pred),
                   "F1": f1_score(te["nlos"], pred),
                   "ROC-AUC": roc_auc_score(te["nlos"], prob)})
print(pd.DataFrame(scores).round(3).to_string(index=False))
print()
print(f"mean accuracy {np.mean([s['accuracy'] for s in scores]):.3f}   "
      f"mean F1 {np.mean([s['F1'] for s in scores]):.3f}")

model = make_model().fit(train[FEATURES], train["nlos"])
print()
print(f"final model trained on all {len(train):,} rows in {time.perf_counter()-t0:.1f}s total")

An F1 near 0.76 means roughly one in four real NLOS signals is missed, and roughly
one in five NLOS calls is actually a clean signal. That is the accuracy you are
about to apply.

A random forest also reports which features it leaned on:

In [ ]:
imp = pd.Series(model.named_steps["clf"].feature_importances_, index=FEATURES)
print(imp.sort_values(ascending=False).round(3).to_string())

## Step 4: Upload your RINEX files

You need two files from your occupation:

- the **observation** file, ending `.YYo` or `.YYO` (`YY` is the year, so `.23O`)
- a **navigation** file, ending `.YYp`/`.YYP` or `.YYn`/`.YYN`

Run the cell, then pick both files in the dialog. A 2-hour observation file is
roughly 90 MB and the upload is the slow part; give it a few minutes.

Get the whole workflow running on the 2 hour file before you try a longer one.

In [ ]:
from google.colab import files
uploaded = files.upload()
for k, v in uploaded.items():
    print(f"{k:44s} {len(v)/1e6:8.1f} MB")

## Step 5: Set your two filenames

Copy the names exactly as they appear above, including the extension.

In [ ]:
OBS_FILE = "rudolph_static_02hr.23O"      # <-- your observation file
NAV_FILE = "emlid-WU1-R_raw_20230913170733.23P"   # <-- your navigation file

import pathlib
for f in (OBS_FILE, NAV_FILE):
    p = pathlib.Path(f)
    print(("found   " if p.exists() else "MISSING "), f)

## Step 6: Extract the features and classify

For every satellite at every epoch this computes five numbers, then asks the
model for a verdict:

| Feature | What it measures |
|---|---|
| `cn0` | signal strength, dB-Hz. Bounced signals lose energy |
| `elev_deg` | satellite's angle above the horizon. Low signals skim obstructions |
| `pr_resid_norm` | how far the measured range sits from what a position solve expects |
| `pr_rate_consistency` | whether Doppler agrees with how fast the range is changing |
| `cn0_std` | how much signal strength wobbles over five epochs |

Expect roughly **half a minute for a 2-hour file** and a couple of minutes for
an 8-hour one.

In [ ]:
import sys
sys.path.insert(0, ".")
from extract_features import read_rinex, make_prcopt, raw_epoch_rows, add_derived_features

t0 = time.perf_counter()
obs, nav = read_rinex(pathlib.Path(OBS_FILE), [pathlib.Path(NAV_FILE)])
rows = raw_epoch_rows(obs, nav, make_prcopt())
df = add_derived_features(pd.DataFrame(rows))
print(f"extracted {len(df):,} (epoch, satellite) rows in {time.perf_counter()-t0:.1f} s")

df["nlos_prob"] = model.predict_proba(df[FEATURES])[:, 1]
df["nlos_pred"] = (df["nlos_prob"] >= 0.5).astype(int)
df.to_csv("nlos_predictions.csv", index=False)

flagged = df["nlos_pred"].mean()
print(f"flagged NLOS: {df['nlos_pred'].sum():,} of {len(df):,} rows ({flagged:.1%})")
df.head()

## Step 7: Plot the probability against satellite elevation

The reasoning behind the model is that low, weak, jumpy signals are the likely
bounces. Check whether it behaves that way on your data.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
for pred, label, color in [(0, "predicted LOS", "#2E4756"), (1, "predicted NLOS", "#B4451F")]:
    sub = df[df["nlos_pred"] == pred]
    ax.scatter(sub["elev_deg"], sub["nlos_prob"], s=6, alpha=0.3,
               c=color, label=f"{label} (n={len(sub):,})")
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xlabel("satellite elevation above horizon (degrees)")
ax.set_ylabel("model probability the signal is NLOS")
ax.set_title("NLOS probability vs elevation")
ax.legend(markerscale=3, frameon=False)
fig.tight_layout()
fig.savefig("nlos_vs_elevation.png", dpi=150)
plt.show()

**Write about what you see.** Does the flagged group sit at lower elevations?
Does it have weaker `cn0`? If the pattern is not there, say so. That is a result
worth reporting.

A second view that often helps:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for pred, label, color in [(0, "predicted LOS", "#2E4756"), (1, "predicted NLOS", "#B4451F")]:
    sub = df[df["nlos_pred"] == pred]
    ax.scatter(sub["elev_deg"], sub["cn0"], s=6, alpha=0.3, c=color, label=label)
ax.set_xlabel("satellite elevation above horizon (degrees)")
ax.set_ylabel("C/N0, signal strength (dB-Hz)")
ax.set_title("Signal strength vs elevation, colored by the model's call")
ax.legend(markerscale=3, frameon=False)
fig.tight_layout()
plt.show()

print(df.groupby("nlos_pred")[list(FEATURES)].mean().round(2).to_string())

## Step 8: Is the model credible here?

The model was trained on **KLTDataset**: a GNSS collection from Kowloon Tong,
Hong Kong, a dense high-rise urban canyon. Ground truth came from a sky-pointing
fisheye camera, so satellites projecting onto open sky were labeled LOS and ones
projecting onto a building were labeled NLOS. 26,522 labeled observations from
four runs on one day, 36% of them NLOS.

You measured its held-out performance yourself in Step 3. For reference, the
random forest scores roughly **0.83 accuracy, 0.75 F1, and ROC-AUC between 0.88
and 0.91** averaged across held-out runs. Your numbers will differ in the third
decimal place depending on the scikit-learn version Colab is running that week. An RBF support-vector machine on the same data
scores within a thousandth of that on F1 while taking twenty-five times longer to
train, which is why this notebook uses the forest.

**Whether any of that transfers to a St. Louis field site is untested.**

Answer these in your report:

1. What fraction of the observations did it flag? Is that plausible for a campus
   building, or does it look like the model is applying Hong Kong expectations to a
   different place?
2. Do the flags line up with anything physical? Use the group means printed above.
   Are the flagged signals actually weaker and lower in the sky?
3. Which of its two mistakes would matter more for a survey: discarding a clean
   signal, or keeping a bounced one? The answer depends on how many satellites you
   have to spare.
4. Read `nlos_prob`, not only the flag. How much of the data sits between 0.4 and
   0.6, where the model is close to undecided?
5. The training labels came from a camera deciding whether each satellite projected
   onto sky or onto a building. That call is unreliable near edges, so the reported
   accuracy is measured against an imperfect reference rather than against truth.
   What does that do to how far you trust the number?

In [ ]:
undecided = df["nlos_prob"].between(0.4, 0.6).mean()
print(f"rows the model is effectively undecided about (0.4 to 0.6): {undecided:.1%}")
print(f"confidently NLOS (>0.9): {(df['nlos_prob'] > 0.9).mean():.1%}")
print(f"confidently LOS  (<0.1): {(df['nlos_prob'] < 0.1).mean():.1%}")

## Step 9: Does acting on the flags improve the position?

Flagging signals is only useful if excluding them makes the position better.

The cell below solves a single-point position at every epoch twice: once using
every satellite, and once with the flagged ones removed. It scores both against
the published NRCan solution for this occupation, which is the closest thing to
truth available here.

Three numbers matter. **Distance to NRCan** is accuracy: how close the average
position sits to the right answer. **Epoch scatter** is precision: how much the
per-epoch solutions jump around. **Epochs solved** matters too, because dropping
satellites can leave an epoch with too few to solve at all.

In [ ]:
import numpy as np
import pyrtklib as prl
from extract_features import epoch_slices, sat_name

# NRCan PPP solution for this occupation, ECEF metres, from its .sum report.
TRUTH = np.array([-26509.6397, -4987781.2374, 3962070.6041])

drop = set(zip(df.loc[df.nlos_pred == 1, "epoch"], df.loc[df.nlos_pred == 1, "sat"]))
print(f"excluding {len(drop):,} of {len(df):,} (epoch, satellite) pairs")


def solve_all_epochs(exclude=None):
    """Per-epoch single-point position. exclude: {(epoch, sat_name)} to drop."""
    out = []
    for e, (i, m) in enumerate(epoch_slices(obs)):
        one = prl.obs_t()
        one.data = prl.Arr1Dobsd_t(m)
        k = 0
        for j in range(m):
            rec = obs.data[i + j]
            if rec.rcv != 1:
                continue
            if exclude and (e, sat_name(rec.sat)) in exclude:
                continue
            one.data[k] = rec
            k += 1
        one.n = one.nmax = k
        if k < 4:
            continue
        sol = prl.sol_t(); sol.time = one.data[0].time
        ssat = prl.Arr1Dssat_t(prl.MAXSAT)
        azel = prl.Arr1Ddouble(one.n * 2); msg = prl.Arr1Dchar(128)
        if prl.pntpos(one.data.ptr, one.n, nav, prcopt, sol, azel, ssat.ptr, msg):
            out.append([sol.rr[0], sol.rr[1], sol.rr[2], k])
    return np.array(out)


prcopt = make_prcopt()
results = {}
for label, excl in (("all satellites", None), ("flagged excluded", drop)):
    P = solve_all_epochs(excl)
    mean = P[:, :3].mean(axis=0)
    results[label] = {
        "epochs solved": len(P),
        "mean sats/epoch": round(float(P[:, 3].mean()), 1),
        "distance to NRCan (m)": round(float(np.linalg.norm(mean - TRUTH)), 3),
        "epoch scatter (m)": round(float(np.linalg.norm(P[:, :3] - mean, axis=1).mean()), 3),
    }

print()
print(pd.DataFrame(results).to_string())

Write up what you see, and answer these.

1. Did excluding the flagged satellites move the position closer to the NRCan
   solution, or further away? By how much?
2. What happened to the epoch scatter, and is that the same question as accuracy
   or a different one?
3. How many satellites per epoch did you lose, and did the number of solvable
   epochs go up or down? Explain the direction it moved.
4. Suppose the position had got worse. What would that have told you, and would it
   mean the flags were wrong?

Two limits worth stating in your report. The position here is a single-frequency
pseudorange solution, which is much cruder than the carrier-phase solutions NRCan
and Emlid Studio produce, so an improvement at this level does not guarantee an
improvement there. And the NRCan solution is itself an estimate, reported with its
own uncertainties, not a surveyed monument.

## Step 10: Put the filtered data back through Emlid Studio

The cell below writes a new RINEX file with the flagged observations removed, so
you can run the same postprocessing you did in Part 2 on filtered data and compare
the two answers.

Download the file it produces, open it in Emlid Studio exactly as before, and
report the position alongside your unfiltered result. If you have time, submit it
to NRCan as well; that turnaround is slower.

Check the epoch count the cell prints against the original before you trust the
output.

In [ ]:
import re

OUT = "rover_filtered.23O"
drop_by_epoch = {}
for e, sat in drop:
    drop_by_epoch.setdefault(e, set()).add(sat)

def sat_id(obs_line):
    """Canonical satellite id from an observation line.

    The file space-pads single digits ("G 5") while sat_name() zero-pads
    ("G05"), so the two must be normalised before comparing or the filter
    silently matches nothing.
    """
    return obs_line[0] + obs_line[1:3].replace(" ", "0")


written = removed = 0
epoch_no = -1
with open(OBS_FILE, errors="replace") as fin, open(OUT, "w") as fout:
    for line in fin:                       # header, copied unchanged
        fout.write(line)
        if "END OF HEADER" in line:
            break
    pending_hdr, block = None, []

    def flush():
        global written, removed
        if pending_hdr is None:
            return
        losers = drop_by_epoch.get(epoch_no, ())
        keep = [b for b in block if sat_id(b) not in losers]
        removed += len(block) - len(keep)
        # satellite count lives in columns 33-35, i.e. slice [32:35]
        fout.write(pending_hdr[:32] + f"{len(keep):3d}" + pending_hdr[35:])
        fout.writelines(keep)
        written += 1

    for line in fin:
        if line.startswith(">"):
            flush()
            pending_hdr, block = line, []
            epoch_no += 1
        elif pending_hdr is not None:
            block.append(line)
    flush()

print(f"wrote {OUT}")
print(f"  epochs        {written:,}")
print(f"  observations removed {removed:,}")

---
### Where this came from

The classifier and the labeled dataset behind it are from:

Hu, R., Wen, W., & Hsu, L.-T. (2023). Fisheye camera aided GNSS NLOS detection
and learning-based pseudorange bias correction for intelligent vehicles in urban
canyons. *2023 IEEE 26th International Conference on Intelligent Transportation
Systems (ITSC)*, Bilbao, Spain, 6088–6095.
<https://doi.org/10.1109/ITSC57777.2023.10422540>

`pyrtklib`, the library doing the position solving in this notebook, has its own
paper, and that one is free to read:

Hu, R., Xu, P., Zhong, Y., & Wen, W. (2024). pyrtklib: An open-source package
for tightly coupled deep learning and GNSS integration for positioning in urban
canyons. arXiv:2409.12996. <https://arxiv.org/abs/2409.12996>